In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import librosa
import os
from IPython.display import Audio
import matplotlib.pyplot as plt
import soundfile as sf
import pickle

In [3]:
from models.codec.kmeans.kmeans_model import KMeans, KMeansEMA
from models.tts.soundstorm.soundstorm_model import SoundStorm
from models.codec.amphion_codec.codec import CodecEncoder, CodecDecoder
from transformers import Wav2Vec2BertModel
import safetensors
from utils.util import load_config

/usr/local/lib/python3.9/dist-packages/bytedmetrics/__init__.py:10: UserWarning: bytedmetrics is renamed to bytedance.metrics, please using `bytedance.metrics` instead of `bytedmetrics`
  warnings.warn("bytedmetrics is renamed to bytedance.metrics, please using `bytedance.metrics` instead of `bytedmetrics`")


In [4]:
cfg = load_config("/opt/tiger/SpeechGeneration/egs/tts/SoundStorm/exp_config_16k_emilia_llama.json")
print(cfg)

{'model_type': 'SoundStorm', 'dataset': ['emilia'], 'preprocess': {'hop_size': 320, 'sample_rate': 16000, 'processed_dir': '', 'valid_file': 'valid.json', 'train_file': 'train.json'}, 'model': {'soundstorm': {'num_quantizer': 8, 'hidden_size': 1024, 'num_layers': 16, 'num_heads': 16, 'codebook_size': 1024, 'cfg_scale': 0.15, 'mask_layer_schedule': 'linear', 'use_cond_code': True, 'cond_codebook_size': 2048, 'cond_dim': 1024, 'use_llama_style': True}, 'kmeans': {'type': 'kmeans_ema', 'stat_mean_var_path': '/mnt/bn/yuacnwang-speech/ckpt/semantic_kmeans/mls_wav2vec2bert_stats.pt', 'kmeans': {'codebook_size': 2048, 'codebook_dim': 1024, 'kmeans_init': True, 'kmeans_iters': 10, 'decay': 0.8, 'eps': 1e-05}, 'pretrained_path': '/mnt/bn/yuacnwang-speech/ckpt/semantic_kmeans/emilia_1k_2048_15k/model.safetensors'}, 'codec': {'encoder': {'d_model': 96, 'up_ratios': [4, 4, 4, 5], 'out_channels': 256, 'use_tanh': False, 'pretrained_path': '/mnt/bn/yuacnwang-speech/ckpt/codec/codec_dac_vocos_16K_320

In [5]:
def build_soundstorm(cfg, pretrained_path, device):
    soundstorm_model = SoundStorm(cfg=cfg.model.soundstorm)
    if ".bin" in pretrained_path:
        soundstorm_model .load_state_dict(torch.load(pretrained_path))
    elif ".safetensors" in pretrained_path:
        safetensors.torch.load_model(soundstorm_model, pretrained_path)
    soundstorm_model.eval()
    soundstorm_model.to(device)
    return soundstorm_model

def build_kmeans_model(cfg, device):
    if cfg.model.kmeans.type == "kmeans":
        kmeans_model = KMeans(cfg=cfg.model.kmeans.kmeans)
    elif cfg.model.kmeans.type == "kmeans_ema":
        kmeans_model = KMeansEMA(cfg=cfg.model.kmeans.kmeans)
    kmeans_model.eval()
    pretrained_path =cfg.model.kmeans.pretrained_path
    if ".bin" in pretrained_path:
        kmeans_model.load_state_dict(torch.load(pretrained_path))
    elif ".safetensors" in pretrained_path:
        safetensors.torch.load_model(kmeans_model, pretrained_path)
    kmeans_model.to(device)
    return kmeans_model

def build_semantic_model(cfg, device):
    semantic_model = Wav2Vec2BertModel.from_pretrained("/mnt/bn/yuacnwang-speech/ckpt/w2v-bert-2")
    semantic_model.eval()
    semantic_model.to(device)
    # layer_idx = 15
    # if layer_idx == 23:
    #     output_idx = 0
    # else:
    #     output_idx = layer_idx + 2
    layer_idx = 15
    output_idx = 17
    stat_mean_var = torch.load(cfg.model.kmeans.stat_mean_var_path)
    semantic_mean = stat_mean_var["mean"]
    semantic_std = torch.sqrt(stat_mean_var["var"])
    semantic_mean = semantic_mean.to(device)
    semantic_std = semantic_std.to(device)
    # print(
    #     "semantic mean: ", semantic_mean, "semantic std: ", semantic_std
    # )
    return semantic_model, semantic_mean, semantic_std

def build_codec_model(cfg, device):
    codec_encoder = CodecEncoder(cfg=cfg.model.codec.encoder)
    codec_decoder = CodecDecoder(cfg=cfg.model.codec.decoder)
    codec_encoder.load_state_dict(
        torch.load(cfg.model.codec.encoder.pretrained_path)
    )
    codec_decoder.load_state_dict(
        torch.load(cfg.model.codec.decoder.pretrained_path)
    )
    # codec_decoder = codec_decoder.quantizer  # we only need the quantizer
    codec_encoder.eval()
    codec_decoder.eval()
    codec_encoder.to(device)
    codec_decoder.to(device)
    return codec_encoder, codec_decoder

In [6]:
device = torch.device("cpu")
soundstorm_pretrained_path = "/mnt/bn/yuacnwang-speech/ckpt/soundstorm/soundstorm_16k_kmeans_2048_emilia_50k_llama/checkpoint/epoch-0003_step-0052000_loss-3.567688/model.safetensors"
soundstorm_model = build_soundstorm(cfg, soundstorm_pretrained_path, device)
semantic_model, semantic_mean, semantic_std = build_semantic_model(cfg, device)
kmeans_model = build_kmeans_model(cfg, device)
codec_encoder, codec_decoder = build_codec_model(cfg, device)

In [7]:
from transformers import SeamlessM4TFeatureExtractor
processor = SeamlessM4TFeatureExtractor.from_pretrained("/mnt/bn/yuacnwang-speech/ckpt/w2v-bert-2")

In [8]:
@torch.no_grad()
def extract_acoustic_code(speech):
    vq_emb = codec_encoder(speech.unsqueeze(1))
    _, vq, _, _, _ = codec_decoder.quantizer(vq_emb)
    acoustic_code = vq.permute(
        1, 2, 0
    )  # (num_quantizer, T, C) -> (T, C, num_quantizer)
    return acoustic_code

@torch.no_grad()
def extract_semantic_code(semantic_mean, semantic_std, input_features, attention_mask):
    vq_emb = semantic_model(
        input_features=input_features,
        attention_mask=attention_mask,
        output_hidden_states=True,
    )
    feat = vq_emb.hidden_states[17]  # (B, T, C)
    feat = (feat - semantic_mean.to(feat)) / semantic_std.to(feat)

    semantic_code, _ = kmeans_model.quantize(feat)  # (B, T)
    return semantic_code

@torch.no_grad()
def extract_features(speech, processor):
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt")
    input_features = inputs["input_features"][0]
    attention_mask = inputs["attention_mask"][0]
    return input_features, attention_mask

In [9]:
dataset_path = "/mnt/bn/yuacnwang-speech/dataset/Emilia/emilia/emilia_50k/wav"
dataset_wav_info = []
for subset in os.listdir(dataset_path):
    for wav in os.listdir(os.path.join(dataset_path, subset)):
        wav_path = os.path.join(dataset_path, subset, wav)
        dataset_wav_info.append(wav_path)
print(len(dataset_wav_info))

1632077


In [10]:
random_idx = 98429
wav_path = dataset_wav_info[random_idx]
meta_path = wav_path.replace("/wav/", "/meta/").replace(".wav", ".pkl")
with open(meta_path, 'rb') as f:
    meta_info = pickle.load(f)
uid = wav_path.split("/")[-1].split(".")[0]
print(uid)
speech, sr = librosa.load(wav_path, sr=16000)
print(meta_info)
Audio(speech, rate=sr)

xmly00000_79015847_697855358_32
{'text': '一个是想坐没人接单。另一个是想坐没人下单。这真是瞌睡遇到了枕头。', 'start': 315.9336061120543, 'end': 324.15060611205433, 'speaker': 'SPEAKER_00', 'language': 'zh', 'phone': 'i↑_k⁼ə↓_s`ɹ`↓_ʃiɑŋ↓↑_ts⁼wo↓_meɪ↑_ɹ`ən↑_tʃ⁼iɛ→_t⁼an→.liŋ↓_i↑_k⁼ə↓_s`ɹ`↓_ʃiɑŋ↓↑_ts⁼wo↓_meɪ↑_ɹ`ən↑_ʃja↓_t⁼an→.ts`⁼ə↓_ts`⁼ən→_s`ɹ`↓_kʰə→_s`weɪ↓_ɥ↓_t⁼ɑʊ↓_lə_ts`⁼ən↓↑_tʰoʊ.', 'phone_mfa': 'i↑ _ k⁼ ə↓ _ s`ɹ`↓ _ ʃ iɑŋ↓↑ _ ts⁼ wo↓ _ m eɪ↑ _ ɹ` ən↑ _ tʃ⁼ iɛ→ _ t⁼ an→ . l iŋ↓ _ i↑ _ k⁼ ə↓ _ s`ɹ`↓ _ ʃ iɑŋ↓↑ _ ts⁼ wo↓ _ m eɪ↑ _ ɹ` ən↑ _ ʃ ja↓ _ t⁼ an→ . ts`⁼ ə↓ _ ts`⁼ ən→ _ s`ɹ`↓ _ kʰ ə→ _ s` weɪ↓ _ ɥ↓ _ t⁼ ɑʊ↓ _ l ə _ ts`⁼ ən↓↑ _ tʰ oʊ .', 'mos': {'wvmos': 0, 'dnsmos': 3.549708461775235, 'avg': 3.549708461775235}, 'uid': 'xmly00000_79015847_697855358_32'}


In [11]:
wav_path = "/mnt/bn/yuacnwang-speech/dataset/temp_test/ns2_3.wav"
uid = wav_path.split("/")[-1].split(".")[0]
print(uid)
speech, sr = librosa.load(wav_path, sr=16000)

ns2_3


In [12]:
input_fetures, attention_mask = extract_features(speech, processor)
input_fetures = input_fetures.unsqueeze(0)
attention_mask = attention_mask.unsqueeze(0)
semantic_code = extract_semantic_code(semantic_mean, semantic_std, input_fetures, attention_mask)

In [13]:
print(semantic_code.shape)

torch.Size([1, 1000])


In [14]:
acoustic_code = extract_acoustic_code(torch.tensor(speech).unsqueeze(0))

In [15]:
print(acoustic_code.shape)

torch.Size([1, 1000, 8])


In [16]:
seq_len = min(semantic_code.shape[1], acoustic_code.shape[1])
semantic_code = semantic_code[:, :seq_len]
acoustic_code = acoustic_code[:, :seq_len, :]

In [17]:
cond = soundstorm_model.cond_emb(semantic_code)
print(cond.shape)

torch.Size([1, 1000, 1024])


In [18]:
prompt = acoustic_code[:,:50*3,:]
predict = soundstorm_model.reverse_diffusion(cond=cond, prompt=prompt, temp=1.5, filter_thres=0.98, n_timesteps=[20, 4, 1, 1, 1, 1, 1, 1], cfg=1.0, rescale_cfg=1.0)
print(predict.shape)

torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850])
torch.Size([1, 850, 8])
torch.Size([1, 850, 8])


In [19]:
vq_emb = codec_decoder.vq2emb(predict.permute(2,0,1))
recovered_audio = codec_decoder(vq_emb)
recovered_audio = recovered_audio[0][0].cpu().detach().numpy()
sf.write("/opt/tiger/SpeechGeneration/temp_test_wav/target/{}.wav".format(uid), recovered_audio, samplerate=16000)
Audio(recovered_audio, rate=16000)

In [20]:
prompt_vq_emb = codec_decoder.vq2emb(prompt.permute(2,0,1))
recovered_prompt_audio = codec_decoder(prompt_vq_emb)
recovered_prompt_audio = recovered_prompt_audio[0][0].cpu().detach().numpy()
sf.write("/opt/tiger/SpeechGeneration/temp_test_wav/prompt/{}.wav".format(uid), recovered_prompt_audio, samplerate=16000)
Audio(recovered_prompt_audio, rate=16000)

In [21]:
combine_audio = np.concatenate([recovered_prompt_audio, recovered_audio])
sf.write("/opt/tiger/SpeechGeneration/temp_test_wav/combine/{}.wav".format(uid), combine_audio, samplerate=16000)
Audio(combine_audio, rate=16000)